# Context-FID Score Presentation
## Necessary packages and functions call

- Context-FID score: A useful metric measures how well the the synthetic time series windows ”fit” into the local context of the time series

In [1]:
import os
import torch
import numpy as np
import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))
from Utils.context_fid import Context_FID
from Utils.metric_utils import display_scores
from Utils.cross_correlation import CrossCorrelLoss


## Data Loading ETTh

Load original dataset and preprocess the loaded data.

In [ ]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
ori_data = np.load('../OUTPUT/etth_energy/samples/ETTh_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etth_energy/ddpm_fake_etth_milestone_500_mask0.0.npy')


print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


# ori_data = ori_data.transpose(0, 2, 1).reshape(b * n, t, 1)

fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (17397, 24, 7)
fake shape is:  (122400, 24, 1)
ori shape is:  (121779, 24, 1)
fake shape is:  (122400, 24, 1)


## Context-FID Score

- The Frechet Inception distance-like score is based on unsupervised time series embeddings. It is able to score the fit of the fixed length synthetic samples into their context of (often much longer) true time series.

- The lowest scoring models correspond to the best performing models in downstream tasks

In [3]:
for j in range(1):

    context_fid_score = []

    for i in range(iterations):
        context_fid = Context_FID(ori_data[:], fake_data[:ori_data.shape[0]])
        context_fid_score.append(context_fid)
        print(f'Iter {i}: ', 'context-fid =', context_fid, '\n')

    display_scores(context_fid_score)

# Seed 12345 Final Score:  0.13663267402399055 ± 0.005536055372540525
# Seed 12345 Final Score:  0.129
# Seed 12345 Final Score:  0.135

# Seed 2025 Final Score:  0.148
# Seed 2025 Final Score:  0.147
# Seed 2025 Final Score:  0.142
# Seed 2025 Final Score:  0.148


#  mp gpt2,   0, 2, 1  reshape , f inal Score:  0.019101439663694805 ± 0.0009810819469392013
# try if remove gpt....  Final Score:  0.02752435505708074 ± 0.0022899093124049484

Iter 0:  context-fid = 0.012297464781163826 

Iter 1:  context-fid = 0.012190693053205918 

Iter 2:  context-fid = 0.013448805826279876 

Iter 3:  context-fid = 0.012731219589022066 

Iter 4:  context-fid = 0.0129636824911081 

Final Score:  0.012726373148155959 ± 0.0006359101094050412


## Correlational Score

- The metric uses the absolute error of the auto-correlation estimator by real data and synthetic data as the metric to assess the temporal dependency.

- For d > 1, it uses the l1-norm of the difference between cross correlation matrices.

In [3]:
def random_choice(size, num_select=100):
    select_idx = np.random.randint(low=0, high=size, size=(num_select,))
    return select_idx

In [4]:
x_real = torch.from_numpy(ori_data)
x_fake = torch.from_numpy(fake_data)

correlational_score = []
size = int(x_real.shape[0] / iterations)

for i in range(iterations):
    real_idx = random_choice(x_real.shape[0], size)
    fake_idx = random_choice(x_fake.shape[0], size)
    corr = CrossCorrelLoss(x_real[real_idx, :, :], name='CrossCorrelLoss')
    loss = corr.compute(x_fake[fake_idx, :, :])
    correlational_score.append(loss.item())
    print(f'Iter {i}: ', 'cross-correlation =', loss.item(), '\n')

display_scores(correlational_score)

ind[0] [tensor(0)]
ind[1] [tensor(0)]
x_l torch.Size([24355, 24, 1])
x_r torch.Size([24355, 24, 1])
ind[0] [tensor(0)]
ind[1] [tensor(0)]
x_l torch.Size([24355, 24, 1])
x_r torch.Size([24355, 24, 1])
Iter 0:  cross-correlation = 2.3314683517128287e-15 

ind[0] [tensor(0)]
ind[1] [tensor(0)]
x_l torch.Size([24355, 24, 1])
x_r torch.Size([24355, 24, 1])
ind[0] [tensor(0)]
ind[1] [tensor(0)]
x_l torch.Size([24355, 24, 1])
x_r torch.Size([24355, 24, 1])
Iter 1:  cross-correlation = 8.881784197001253e-17 

ind[0] [tensor(0)]
ind[1] [tensor(0)]
x_l torch.Size([24355, 24, 1])
x_r torch.Size([24355, 24, 1])
ind[0] [tensor(0)]
ind[1] [tensor(0)]
x_l torch.Size([24355, 24, 1])
x_r torch.Size([24355, 24, 1])
Iter 2:  cross-correlation = 7.105427357601002e-16 

ind[0] [tensor(0)]
ind[1] [tensor(0)]
x_l torch.Size([24355, 24, 1])
x_r torch.Size([24355, 24, 1])
ind[0] [tensor(0)]
ind[1] [tensor(0)]
x_l torch.Size([24355, 24, 1])
x_r torch.Size([24355, 24, 1])
Iter 3:  cross-correlation = 1.931788062

## Data Loading energy

Load original dataset and preprocess the loaded data.

In [7]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
ori_data = np.load('../OUTPUT/etth_energy/samples/energy_data_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etth_energy/ddpm_fake_energy_milestone_500_mask0.0.npy')



print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(0, 2, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (19712, 24, 28)
fake shape is:  (552600, 24, 1)
ori shape is:  (551936, 24, 1)
fake shape is:  (552600, 24, 1)


## Context-FID Score

- The Frechet Inception distance-like score is based on unsupervised time series embeddings. It is able to score the fit of the fixed length synthetic samples into their context of (often much longer) true time series.

- The lowest scoring models correspond to the best performing models in downstream tasks

In [8]:
for j in range(1):

    context_fid_score = []

    for i in range(iterations):
        context_fid = Context_FID(ori_data[:], fake_data[:ori_data.shape[0]])
        context_fid_score.append(context_fid)
        print(f'Iter {i}: ', 'context-fid =', context_fid, '\n')

    display_scores(context_fid_score)

# Seed 12345 Final Score:  0.13663267402399055 ± 0.005536055372540525
# Seed 12345 Final Score:  0.129
# Seed 12345 Final Score:  0.135

# Seed 2025 Final Score:  0.148
# Seed 2025 Final Score:  0.147
# Seed 2025 Final Score:  0.142
# Seed 2025 Final Score:  0.148


#  mp gpt2,   0, 2, 1  reshape , f inal Score:  0.019101439663694805 ± 0.0009810819469392013
# try if remove gpt....  Final Score:  0.02752435505708074 ± 0.0022899093124049484

Iter 0:  context-fid = 0.028250207156578534 

Iter 1:  context-fid = 0.030662997037942494 

Iter 2:  context-fid = 0.032668257967886374 

Iter 3:  context-fid = 0.034466191502857554 

Iter 4:  context-fid = 0.03253300498859822 

Final Score:  0.03171613173077263 ± 0.002929010160121354


## Correlational Score

- The metric uses the absolute error of the auto-correlation estimator by real data and synthetic data as the metric to assess the temporal dependency.

- For d > 1, it uses the l1-norm of the difference between cross correlation matrices.

In [13]:
def random_choice(size, num_select=100):
    select_idx = np.random.randint(low=0, high=size, size=(num_select,))
    return select_idx

In [14]:
x_real = torch.from_numpy(ori_data)
x_fake = torch.from_numpy(fake_data)

correlational_score = []
size = int(x_real.shape[0] / iterations)

for i in range(iterations):
    real_idx = random_choice(x_real.shape[0], size)
    fake_idx = random_choice(x_fake.shape[0], size)
    corr = CrossCorrelLoss(x_real[real_idx, :, :], name='CrossCorrelLoss')
    loss = corr.compute(x_fake[fake_idx, :, :])
    correlational_score.append(loss.item())
    print(f'Iter {i}: ', 'cross-correlation =', loss.item(), '\n')

display_scores(correlational_score)

Iter 0:  cross-correlation = 1.5099033134902129e-15 

Iter 1:  cross-correlation = 2.4424906541753446e-16 

Iter 2:  cross-correlation = 1.006972283335017e-14 

Iter 3:  cross-correlation = 8.72635297355373e-15 

Iter 4:  cross-correlation = 1.0391687510491465e-14 

Final Score:  6.188383139260622e-15 ± 6.095380283124244e-15
